# Standalone valuation and implied synergy

Values Organon on a standalone basis as of the unaffected date (9-Apr-2026, the last close before deal speculation moved the price), then solves for the revenue growth path that the $14.00 offer requires - the implied-synergy question.

In [1]:

import sys, warnings
sys.path.insert(0, r"/Users/shaan/Desktop/FAM/sunpharma-organon-merger-arbitrage")
warnings.filterwarnings("ignore")

from src import config as cfg, data, statements, valuation, checks
import pandas as pd
import numpy as np
pd.set_option("display.width", 160)

prices = data.load_prices()
ogn_stmts = data.load_statements(cfg.TARGET)


## Beta: pre-leak estimation window

The standard (-250,-30) trading day window relative to the 26-Apr announcement would sit on top of the 16-Jan-to-26-Apr run-up and contaminate both alpha and beta. The window here ends the day before the leak date and runs back 250 trading days.

In [2]:

ogn_close = prices[cfg.TARGET]["Close"]
mkt_close = prices[cfg.US_INDEX]["Close"]
ogn_ret = np.log(ogn_close / ogn_close.shift(1)).dropna()
mkt_ret = np.log(mkt_close / mkt_close.shift(1)).dropna()

window_end = pd.Timestamp(cfg.ESTIMATION_WINDOW_END)
checks.check_beta_window_uncontaminated(cfg.ESTIMATION_WINDOW_END)

est_dates = ogn_ret.index[ogn_ret.index <= window_end][-cfg.ESTIMATION_WINDOW_LENGTH:]
beta, alpha_ann, n_obs, r2 = valuation.regression_beta(ogn_ret.loc[est_dates], mkt_ret.loc[est_dates])

print(f"estimation window : {est_dates[0].date()} to {est_dates[-1].date()}  (n={n_obs})")
print(f"beta               : {beta:.3f}")
print(f"annualised alpha   : {alpha_ann:+.2%}   (pre-event trajectory - the business was already declining)")
print(f"R-squared          : {r2:.3f}")


estimation window : 2025-01-17 to 2026-01-15  (n=250)
beta               : 0.969
annualised alpha   : -77.85%   (pre-event trajectory - the business was already declining)
R-squared          : 0.076


## Cost of equity

Risk-free rate is the 10-year US Treasury yield on the unaffected date. The equity risk premium is Damodaran's implied ERP for the United States, 4.45% as of the July 2026 data update - a published, dated, sourced figure rather than an assumed one.

In [3]:

tnx = prices[cfg.UST10Y]["Close"]
d_unaffected = pd.Timestamp(cfg.DEAL["unaffected_date"])
rf = tnx.loc[tnx.index <= d_unaffected].iloc[-1] / 100

ERP_SOURCE = "Damodaran, Equity Risk Premiums: Determinants, Estimates and Implications, 2026 edition (implied ERP, July 2026 update)"
erp = 0.0445

ke = valuation.cost_of_equity(rf, beta, erp)
print(f"risk-free rate (10Y UST, {d_unaffected.date()}) : {rf:.3%}")
print(f"equity risk premium ({ERP_SOURCE})")
print(f"                                          : {erp:.2%}")
print(f"cost of equity (CAPM)                     : {ke:.3%}")


risk-free rate (10Y UST, 2026-04-09) : 4.293%
equity risk premium (Damodaran, Equity Risk Premiums: Determinants, Estimates and Implications, 2026 edition (implied ERP, July 2026 update))
                                          : 4.45%
cost of equity (CAPM)                     : 8.607%


## Cost of debt: embedded vs. marginal

Organon's FY2025 interest expense divided by its average debt gives the *embedded* rate on debt issued mostly at 2021 spinoff-era, investment-grade-adjacent terms. That is not the rate at which Organon could borrow *today*, given how its credit profile has since deteriorated. Interest coverage (EBIT / interest expense) of 1.84x maps, under Damodaran's synthetic-rating framework (interest coverage to synthetic rating and default spread, smaller-firm table), to a B3/B- rating and a 5.50% default spread over the risk-free rate - a marginal pretax cost of debt near 9.8%, materially above the embedded 5.75%. The marginal rate is the economically correct one for a forward-looking DCF: it reflects what new capital would actually cost this credit today, not what a healthier prior version of the company locked in years ago.

In [4]:

inc = ogn_stmts["income_stmt"]
bs = ogn_stmts["balance_sheet"].reindex(columns=inc.columns)
cf = ogn_stmts["cashflow"].reindex(columns=inc.columns)

fy2025 = inc.columns[0]
fy2024 = inc.columns[1]
interest_expense_fy25 = abs(inc.loc["Interest Expense", fy2025])
avg_debt = (bs.loc["Total Debt", fy2025] + bs.loc["Total Debt", fy2024]) / 2
tax_rate = inc.loc["Tax Rate For Calcs", fy2025]
interest_coverage_fy25 = inc.loc["EBIT", fy2025] / interest_expense_fy25

kd_embedded_pretax = interest_expense_fy25 / avg_debt
kd_embedded_aftertax = valuation.cost_of_debt(interest_expense_fy25, avg_debt, tax_rate)

# Damodaran synthetic rating table (smaller firms, mkt cap < $5bn):
# interest coverage 1.5x-2.0x -> B3/B- -> 5.50% default spread over Rf
SYNTHETIC_RATING_SOURCE = "Damodaran, Ratings, Interest Coverage Ratios and Default Spreads (smaller-firm table)"
default_spread = 0.0550
kd_marginal_pretax = rf + default_spread
kd_marginal_aftertax = kd_marginal_pretax * (1 - tax_rate)

print(f"FY2025 interest coverage (EBIT / interest exp.) : {interest_coverage_fy25:.2f}x  -> synthetic rating B3/B-")
print(f"\n{'':25}{'embedded':>14}{'marginal (synthetic)':>22}")
print(f"{'pre-tax cost of debt':25}{kd_embedded_pretax:>14.3%}{kd_marginal_pretax:>22.3%}")
print(f"{'after-tax cost of debt':25}{kd_embedded_aftertax:>14.3%}{kd_marginal_aftertax:>22.3%}")
print(f"\nsource: {SYNTHETIC_RATING_SOURCE}")


FY2025 interest coverage (EBIT / interest exp.) : 1.84x  -> synthetic rating B3/B-

                               embedded  marginal (synthetic)
pre-tax cost of debt             5.752%                9.793%
after-tax cost of debt           4.544%                7.736%

source: Damodaran, Ratings, Interest Coverage Ratios and Default Spreads (smaller-firm table)


In [5]:

shares = cfg.DEAL["shares_outstanding"]
market_cap_unaffected = shares * cfg.DEAL["offer_price_usd"] / (1 + 1.029)  # back out the unaffected price via the reconciled 102.9% premium
unaffected_price = market_cap_unaffected / shares
net_debt_fy25 = bs.loc["Total Debt", fy2025] - bs.loc["Cash And Cash Equivalents", fy2025]

w_embedded = valuation.wacc(market_cap_unaffected, net_debt_fy25, ke, kd_embedded_aftertax)
w_marginal = valuation.wacc(market_cap_unaffected, net_debt_fy25, ke, kd_marginal_aftertax)

print(f"market cap at unaffected price : ${market_cap_unaffected/1e9:.3f}B")
print(f"net debt (FY2025)              : ${net_debt_fy25/1e9:.3f}B")
print(f"equity weight                  : {market_cap_unaffected/(market_cap_unaffected+net_debt_fy25):.1%}")
print(f"WACC (embedded cost of debt)   : {w_embedded:.3%}")
print(f"WACC (marginal/synthetic)      : {w_marginal:.3%}")


market cap at unaffected price : $1.812B
net debt (FY2025)              : $8.070B
equity weight                  : 18.3%
WACC (embedded cost of debt)   : 5.289%
WACC (marginal/synthetic)      : 7.896%


## Explicit FCFF forecast

Built from FY2025 actuals, not assumed lump growth. Revenue held flat (base case) - the three-year average is roughly flat and FY2025 itself was down 2.9% - with EBIT margin, D&A, capex and working-capital investment each held at their FY2025 ratio to revenue. This is deliberately conservative: it assumes the margin compression seen from FY2022 (25.0%) to FY2025 (14.95%) stops rather than continues.

In [6]:

revenue_fy25 = inc.loc["Total Revenue", fy2025]
ebit_margin_fy25 = inc.loc["EBIT", fy2025] / revenue_fy25
da_pct = cf.loc["Depreciation And Amortization", fy2025] / revenue_fy25
capex_pct = cf.loc["Capital Expenditure", fy2025] / revenue_fy25   # negative
dwc_pct = cf.loc["Change In Working Capital", fy2025] / revenue_fy25  # negative

print(f"FY2025 revenue        : ${revenue_fy25/1e9:.3f}B")
print(f"FY2025 EBIT margin    : {ebit_margin_fy25:.2%}")
print(f"D&A % of revenue      : {da_pct:.2%}")
print(f"CapEx % of revenue    : {capex_pct:.2%}")
print(f"Delta NWC % of revenue: {dwc_pct:.2%}")

def forecast_fcff(revenue_growth, n_years=5, margin=ebit_margin_fy25, tax=tax_rate):
    revs, fcffs = [], []
    rev = revenue_fy25
    for _ in range(n_years):
        rev = rev * (1 + revenue_growth)
        ebit = rev * margin
        nopat = ebit * (1 - tax)
        fcff = nopat + rev * da_pct + rev * capex_pct + rev * dwc_pct
        revs.append(rev); fcffs.append(fcff)
    return revs, fcffs

base_revs, base_fcffs = forecast_fcff(0.0)
forecast_table = pd.DataFrame({"revenue": base_revs, "FCFF": base_fcffs},
                               index=[f"Y{i+1}" for i in range(5)])
forecast_table.round(1)


FY2025 revenue        : $6.216B
FY2025 EBIT margin    : 14.95%
D&A % of revenue      : 5.81%
CapEx % of revenue    : -5.08%
Delta NWC % of revenue: -3.97%


,revenue,FCFF
Y1,6.216000e+09,531910000.0
Y2,6.216000e+09,531910000.0
Y3,6.216000e+09,531910000.0
Y4,6.216000e+09,531910000.0
Y5,6.216000e+09,531910000.0


## Which discount rate to use

The two modeled WACCs bracket a wide range, and equity value here is a small, highly geared residual of a much larger enterprise value - Organon's net debt ($8.07B) is more than four times its unaffected-price market cap ($1.81B), so small WACC changes swing equity value per share by far more than a proportional amount. Rather than pick one modeled WACC and present a fragile point estimate, the discount rate the market itself was already applying pre-deal is backed out by solving for the WACC that reproduces the actual unaffected price ($6.90) under the flat-revenue base-case forecast above, using the same 1.5% terminal growth used throughout. If that market-implied rate falls inside the embedded-to-marginal range, it is independent evidence the modeling framework is reasonable - and it becomes the rate used from here on, since it is calibrated to observed pricing rather than to a discount-rate assumption.

In [7]:

from scipy.optimize import brentq

TERMINAL_GROWTH = 0.015

def price_at_wacc(disc_rate):
    res = valuation.dcf_value(base_fcffs, disc_rate, TERMINAL_GROWTH, terminal_method="gordon")
    return valuation.equity_value_per_share(res["enterprise_value"], net_debt_fy25, shares)

w_implied = brentq(lambda x: price_at_wacc(x) - unaffected_price, 0.02, 0.30, xtol=1e-6)

print(f"WACC (embedded cost of debt)        : {w_embedded:.3%}")
print(f"WACC (market-implied, from $6.90)   : {w_implied:.3%}   <- used from here on")
print(f"WACC (marginal/synthetic-rating)    : {w_marginal:.3%}")
print()
in_range = w_embedded <= w_implied <= w_marginal
print(f"market-implied rate falls inside the modeled [embedded, marginal] range : {in_range}")

w = w_implied


WACC (embedded cost of debt)        : 5.289%
WACC (market-implied, from $6.90)   : 6.607%   <- used from here on
WACC (marginal/synthetic-rating)    : 7.896%

market-implied rate falls inside the modeled [embedded, marginal] range : True


## Standalone DCF

In [8]:

dcf = valuation.dcf_value(base_fcffs, w, TERMINAL_GROWTH, terminal_method="gordon")

checks.check_dcf_identity(dcf["pv_explicit_sum"], dcf["pv_terminal"], dcf["enterprise_value"])
tv_check = checks.check_terminal_value_share(dcf["pv_explicit_sum"], dcf["pv_terminal"])

print(f"PV of explicit FCFF (Y1-Y5) : ${dcf['pv_explicit_sum']/1e9:.3f}B")
print(f"Terminal value               : ${dcf['terminal_value']/1e9:.3f}B")
print(f"PV of terminal value         : ${dcf['pv_terminal']/1e9:.3f}B")
print(f"Enterprise value              : ${dcf['enterprise_value']/1e9:.3f}B")
print(f"Terminal value as % of EV     : {tv_check['tv_share']:.1%}  {'(FLAGGED: >75%)' if tv_check['flagged'] else ''}")

standalone_value_per_share = valuation.equity_value_per_share(dcf["enterprise_value"], net_debt_fy25, shares)
print(f"\nStandalone equity value/share : ${standalone_value_per_share:.2f}   (matches $6.90 by construction - see above)")
print(f"Offer price                    : ${cfg.DEAL['offer_price_usd']:.2f}")


PV of explicit FCFF (Y1-Y5) : $2.204B
Terminal value               : $10.572B
PV of terminal value         : $7.678B
Enterprise value              : $9.882B
Terminal value as % of EV     : 77.7%  (FLAGGED: >75%)

Standalone equity value/share : $6.90   (matches $6.90 by construction - see above)
Offer price                    : $14.00


## Sensitivity: WACC x terminal growth

Centred on the market-implied WACC. This grid is the honest headline output of this section - a single point estimate would misrepresent how sensitive equity value is to assumptions in a capital structure this levered.

In [9]:

wacc_range = [w - 0.02, w - 0.01, w, w + 0.01, w + 0.02]
growth_range = [0.0, 0.01, TERMINAL_GROWTH, 0.02, 0.03]
grid = valuation.sensitivity_grid(base_fcffs, wacc_range, growth_range, net_debt_fy25, shares)
grid.round(2)


,0.0%,1.0%,1.5%,2.0%,3.0%
4.6%,13.24,23.42,30.97,41.41,81.80
5.6%,5.40,11.70,16.00,21.50,38.82
6.6%,-0.07,4.16,6.90,10.23,19.67
7.6%,-4.10,-1.10,0.78,2.98,8.83
8.6%,-7.20,-4.97,-3.63,-2.08,1.85


## Reverse DCF: what does $14.00 require?

Solves for the constant explicit-period revenue growth rate that makes the standalone DCF equal the offer price, at the market-implied WACC and holding margin and terminal growth at the base-case values above. No synergy figure was disclosed in Sun Pharma's SEC filings (confirmed directly against the 8-K Exhibit 99.1 press release, which uses only qualitative language: \"scope for synergies including significant revenue upside opportunities\"). This is this project's own estimate, not a management figure.

In [10]:

# NOTE: this solves for a REVENUE growth rate fed through the same
# margin / D&A / capex / working-capital ratios used above, not a direct
# FCFF growth rate - so the implied path stays internally consistent with
# the explicit forecast rather than assuming FCFF grows in isolation.
def price_at_growth(g):
    _, fcffs_g = forecast_fcff(g)
    res = valuation.dcf_value(fcffs_g, w, TERMINAL_GROWTH, terminal_method="gordon")
    return valuation.equity_value_per_share(res["enterprise_value"], net_debt_fy25, shares)

implied_growth = brentq(lambda g: price_at_growth(g) - cfg.DEAL["offer_price_usd"], -0.20, 0.50, xtol=1e-6)
implied_value_check = price_at_growth(implied_growth)

hist_2yr_cagr = (inc.loc["Total Revenue"].iloc[0] / inc.loc["Total Revenue"].iloc[2]) ** 0.5 - 1

print(f"Implied constant annual revenue growth to justify $14.00 : {implied_growth:.2%}")
print(f"(check) DCF value at that growth rate                    : ${implied_value_check:.2f}")
print(f"\nFor comparison, FY2023-FY2025 actual revenue CAGR        : {hist_2yr_cagr:.2%}")


Implied constant annual revenue growth to justify $14.00 : 3.87%
(check) DCF value at that growth rate                    : $14.00

For comparison, FY2023-FY2025 actual revenue CAGR        : -0.38%


In [11]:

_, fcffs_implied = forecast_fcff(implied_growth)
implied_ev = valuation.dcf_value(fcffs_implied, w, TERMINAL_GROWTH, terminal_method="gordon")["enterprise_value"]
standalone_ev = dcf["enterprise_value"]
implied_synergy_value = implied_ev - standalone_ev

print(f"Standalone EV (flat-revenue base case, market-implied WACC) : ${standalone_ev/1e9:.3f}B")
print(f"EV implied by the $14.00 offer                               : ${implied_ev/1e9:.3f}B")
print(f"Implied synergy / growth premium                              : ${implied_synergy_value/1e9:.3f}B  "
      f"({implied_synergy_value/standalone_ev:.1%} of standalone EV)")


Standalone EV (flat-revenue base case, market-implied WACC) : $9.882B
EV implied by the $14.00 offer                               : $11.747B
Implied synergy / growth premium                              : $1.865B  (18.9% of standalone EV)


## Summary

In [12]:

summary = pd.Series({
    "beta_pre_leak": beta,
    "risk_free_rate": rf,
    "equity_risk_premium": erp,
    "cost_of_equity": ke,
    "wacc_embedded": w_embedded,
    "wacc_market_implied": w_implied,
    "wacc_marginal_synthetic": w_marginal,
    "wacc_used": w,
    "standalone_value_per_share": standalone_value_per_share,
    "unaffected_market_price": unaffected_price,
    "offer_price": cfg.DEAL["offer_price_usd"],
    "tv_share_of_ev": tv_check["tv_share"],
    "implied_revenue_growth_required": implied_growth,
    "historical_2yr_revenue_cagr": hist_2yr_cagr,
    "implied_synergy_value_usd_bn": implied_synergy_value / 1e9,
})
summary.to_csv(cfg.DATA_FINAL / "valuation_summary.csv", header=["value"])

# the two third-party reference inputs behind the discount rate are recorded
# separately with their source and retrieval date - they are the only figures
# in this project that come from outside the market/filing data pulled directly
pd.DataFrame([
    {"input": "US implied equity risk premium", "value": erp,
     "source": ERP_SOURCE, "retrieved": cfg.SNAPSHOT_DATE},
    {"input": "Default spread, B3/B- synthetic rating", "value": default_spread,
     "source": SYNTHETIC_RATING_SOURCE, "retrieved": cfg.SNAPSHOT_DATE},
]).to_csv(cfg.DATA_EXTERNAL / "cost_of_capital_reference_inputs.csv", index=False)

summary.round(4)


beta_pre_leak                       0.9694
risk_free_rate                      0.0429
equity_risk_premium                 0.0445
cost_of_equity                      0.0861
wacc_embedded                       0.0529
wacc_market_implied                 0.0661
wacc_marginal_synthetic             0.0790
wacc_used                           0.0661
standalone_value_per_share          6.9000
unaffected_market_price             6.9000
offer_price                        14.0000
tv_share_of_ev                      0.7770
implied_revenue_growth_required     0.0387
historical_2yr_revenue_cagr        -0.0038
implied_synergy_value_usd_bn        1.8645
dtype: float64